In [253]:
import pandas as pd
import numpy as np

## Loading my 'anchor' data
Based on preliminary analysis, `job_exposure.csv` (03/05/26 labor_market_impact release) was chosen as the primary data source for merging. It contains occupation codes (`occ_code`), occupation names, and AI exposure scores calculated by the Anthropic team.

In [254]:
job_exposure=pd.read_csv('../dataset/labor_market_impacts/job_exposure.csv')
job_exposure

,occ_code,title,observed_exposure
0,11-1011,Chief Executives,0.0333
1,11-1021,General and Operations Managers,0.1378
2,11-1031,Legislators,0.0000
3,11-2011,Advertising and Promotions Managers,0.1731
4,11-2021,Marketing Managers,0.3195
...,...,...,...
751,53-7071,Gas Compressor and Gas Pumping Station Operators,0.0000
752,53-7072,"Pump Operators, Except Wellhead Pumpers",0.0000
753,53-7073,Wellhead Pumpers,0.0000
754,53-7081,Refuse and Recyclable Material Collectors,0.0000


In [255]:
job_exposure.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 756 entries, 0 to 755
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   occ_code           756 non-null    object 
 1   title              756 non-null    object 
 2   observed_exposure  756 non-null    float64
dtypes: float64(1), object(2)
memory usage: 17.8+ KB


## Adding wage and labor market features by merging `wage_data.csv`
This section adds more features from `wage_data.csv` (02/10/2025 release) to the original occupations dataset.

In [256]:
wage_data=pd.read_csv('../dataset/release_2025_02_10/wage_data.csv')
wage_data['SOCcode']=wage_data['SOCcode'].str.split('.').str[0]
wage_data.drop(columns=['JobName', 'WageGroup'], inplace=True)

# removing duplicates from the data after parsing SOCcodes by aggregating the data
wage_data=wage_data.groupby('SOCcode').agg({
    'JobFamily':'first',
    'isBright':'max',
    'isGreen':'max',
    'JobZone':'mean',
    'MedianSalary':'mean',
    'JobForecast':'sum',
    'ChanceAuto':'mean'
}
).reset_index()
wage_data.rename(columns={'SOCcode':'occ_code'}, inplace=True)

# merging anchor data with wage data
data=job_exposure.merge(wage_data, how='left', on='occ_code')
data=data[['occ_code', 'title', 'JobFamily','isBright','isGreen','JobZone','MedianSalary','JobForecast','ChanceAuto', 'observed_exposure']]

In [257]:
wage_data

,occ_code,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,ChanceAuto
0,11-1011,Management,False,True,5.0,189600.0,33600,0.25
1,11-1021,Management,True,True,4.0,100930.0,230000,16.00
2,11-1031,Management,False,False,4.0,24670.0,4300,-1.00
3,11-2011,Management,False,True,1.5,117130.0,5400,1.50
4,11-2021,Management,True,True,4.0,134290.0,26000,1.40
...,...,...,...,...,...,...,...,...
816,53-7073,Transportation and Material Moving,False,False,2.0,53490.0,1700,84.00
817,53-7081,Transportation and Material Moving,True,True,2.0,37260.0,20200,93.00
818,53-7111,Transportation and Material Moving,False,False,2.0,56340.0,100,37.00
819,53-7121,Transportation and Material Moving,False,False,2.0,38220.0,1200,72.00


In [258]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 756 entries, 0 to 755
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   occ_code           756 non-null    object 
 1   title              756 non-null    object 
 2   JobFamily          672 non-null    object 
 3   isBright           672 non-null    object 
 4   isGreen            672 non-null    object 
 5   JobZone            672 non-null    float64
 6   MedianSalary       672 non-null    float64
 7   JobForecast        672 non-null    float64
 8   ChanceAuto         672 non-null    float64
 9   observed_exposure  756 non-null    float64
dtypes: float64(5), object(5)
memory usage: 59.2+ KB


In [259]:
print(f'Data shape: {data.shape}')
print(f'Number of unique occupation codes: {data.title.nunique()}')
print(f'Number of unique occupation titles: {data.title.nunique()}')
print(f'Duplicates: {data.duplicated().sum()}\n')
print(f'Null values:\n{data.isnull().sum()}')

Data shape: (756, 10)
Number of unique occupation codes: 756
Number of unique occupation titles: 756
Duplicates: 0

Null values:
occ_code              0
title                 0
JobFamily            84
isBright             84
isGreen              84
JobZone              84
MedianSalary         84
JobForecast          84
ChanceAuto           84
observed_exposure     0
dtype: int64


## Handling null values
To ensure high-quality of the data, I will impute the missiing values with median (for quantitative variables) and mode (for qualitative) by major occupation group (first 2 values in the `occ_code`) and JobFamily. This will help me prevent outliers/irrational values skewing my data and confusing the model

In [260]:
data[data.isnull().any(axis=1)]

,occ_code,title,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,ChanceAuto,observed_exposure
6,11-2032,Public Relations Managers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.2315
7,11-3012,Administrative Services Managers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000
8,11-3013,Facilities Managers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1345
32,11-9171,Funeral Home Managers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0108
33,11-9179,"Personal Service Managers, All Other",NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1244
...,...,...,...,...,...,...,...,...,...,...
723,53-3052,"Bus Drivers, Transit and Intercity",NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000
724,53-3053,Shuttle Drivers and Chauffeurs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000
725,53-3054,Taxi Drivers,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000
728,53-4022,"Railroad Brake, Signal, and Switch Operators a...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000


In [261]:
# I will now impure missing job features using the median of each SOC major group (first 2 digits of occ_code)
data['major_group']=data['occ_code'].str[:2]

# numeric columns to impute
num_cols=['JobZone', 'MedianSalary', 'JobForecast', 'ChanceAuto']

for col in num_cols:
    data[col]=data.groupby('major_group')[col].transform(lambda x: x.fillna(x.median()))

# binary columns — impute with mode values from major group
for col in ['isBright', 'isGreen']:
    data[col] = data.groupby('major_group')[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)
    ).astype(int)

# JobFamily — fill from the major group mapping
data['JobFamily'] = data.groupby('major_group')['JobFamily'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown')
)


data[data.isnull().any(axis=1)]

/var/folders/lw/f5qy2qvj20qfwyw16w6_dp7m0000gn/T/ipykernel_89889/3108071107.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)
/var/folders/lw/f5qy2qvj20qfwyw16w6_dp7m0000gn/T/ipykernel_89889/3108071107.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 0)


,occ_code,title,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,ChanceAuto,observed_exposure,major_group


Before modeling, I address missing and invalid values across three variables:

- **JobZone:** Two occupations had a value of −1. I looked up their typical education requirements and manually assigned the correct JobZone.
- **ChanceAuto:** Negative values (−1 and other negatives) represent missing data. I replaced them with NaN and imputed using the median by `JobFamily`, which is more accurate than imputing by major occupation group alone.
- **MedianSalary:** Some occupations had values under $1,000, indicating hourly wages rather than annual salary. I converted these to annual salary assuming 40 hours/week, 52 weeks/year.

In [262]:
# changing negative (missing) JobZone Values
data.loc[data.occ_code == '15-2099', 'JobZone'] = 4
data.loc[data.occ_code == '29-1129', 'JobZone'] = 4
data[data.JobZone<=0]

,occ_code,title,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,ChanceAuto,observed_exposure,major_group


In [263]:
# changing negative (missing/invalid) ChanceAuto values with the median of corresponding JobFamily group
data['ChanceAuto'] = data['ChanceAuto'].where(data['ChanceAuto'] >= 0, np.nan)
data['ChanceAuto'] = data.groupby('JobFamily')['ChanceAuto'].transform(
    lambda x: x.fillna(x.median())
)
data[data.ChanceAuto<=0]

,occ_code,title,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,ChanceAuto,observed_exposure,major_group
98,17-2111,"Health and Safety Engineers, Except Mining Saf...",Architecture and Engineering,0,1,2.75,89130.0,8000.0,0.0,0.0000,17
135,19-2041,"Environmental Scientists and Specialists, Incl...","Life, Physical, and Social Science",1,1,4.75,71130.0,41200.0,0.0,0.0548,19


In [264]:
# converting outliers — likely hourly wages — to yearly salaries, assuming typical work time conditions (40 hr/week, 52 weeks/year )
data.loc[data.MedianSalary<1000, 'MedianSalary']=data.loc[data.MedianSalary<1000, 'MedianSalary']*40*52
data[data.MedianSalary<1000]

,occ_code,title,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,ChanceAuto,observed_exposure,major_group


## Loading task-level data for feature engineering

`onet_task_statements.csv` (02/10/25 release) contains ~19,000 O*NET tasks, `occ_code` associated with each. 

In [265]:
onet_task_statements=pd.read_csv('../dataset/release_2025_02_10/onet_task_statements.csv')
onet_task_statements.columns=onet_task_statements.columns.str.lower().str.replace(' ','_')
onet_task_statements.drop(columns=['title','task_id','incumbents_responding','date','domain_source', 'task_type'], inplace=True)
onet_task_statements.rename(columns={'task':'task_name','o*net-soc_code':'occ_code'}, inplace=True)
onet_task_statements['task_name']=onet_task_statements['task_name'].str.lower()
onet_task_statements['occ_code']=onet_task_statements['occ_code'].str.split('.').str[0]
onet_task_statements['task_name']=onet_task_statements['task_name'].str.strip().str.lower()
onet_task_statements

,occ_code,task_name
0,11-1011,direct or coordinate an organization's financi...
1,11-1011,appoint department heads or managers and assig...
2,11-1011,analyze operations to evaluate performance of ...
3,11-1011,"direct, plan, or implement policies, objective..."
4,11-1011,"prepare budgets for approval, including those ..."
...,...,...
19525,53-7121,"test vessels for leaks, damage, and defects, a..."
19526,53-7121,unload cars containing liquids by connecting h...
19527,53-7121,copy and attach load specifications to loaded ...
19528,53-7121,start pumps and adjust valves or cables to reg...


In [266]:
task_features_by_occ=onet_task_statements

In [267]:
# dropping absolute duplicates – same occ code and task name (because different occupations can have the same task) 
task_features_by_occ.drop_duplicates(subset=['task_name', 'occ_code'], keep='first', inplace=True)
print(f'Number of rows: {task_features_by_occ.shape[0]}')
print(f'Number of unique tasks: {task_features_by_occ.task_name.nunique()}')

Number of rows: 19522
Number of unique tasks: 18428


In [268]:
task_features_by_occ.groupby('occ_code').size().describe()

count    775.000000
mean      25.189677
std       23.805172
min        4.000000
25%       15.000000
50%       20.000000
75%       26.000000
max      290.000000
dtype: float64

## Feature engineering: task-level keyword flags

Now, for each of the 19,522 O*NET tasks (18,428 of which are unique), I create three binary (0/1) indicator columns based on keyword matching against the task description:

- **`has_computer`** — task involves computing, software, or data work
- **`is_physical`** — task requires physical presence, manual labor, or equipment operation
- **`must_communicate`** — task involves interpersonal interaction, counseling, teaching, or human judgment
- **`must_analyze`** — task requires analytical reasoning, research, evaluation, or working with numbers
- **`must_manage`** — task involves overseeing people, projects, budgets, or organizational operations
- **`is_creative`** – task involves creative activity such as that associated with arts, photography, film, design
- **`is_text_native`** - task where the output is inherently textual — writing code, drafting documents, entering data, composing emails
- **`requires_presence`** - task requires being physically with a human, even though the work itself is cognitive

Each flag checks whether the task text contains any word from a curated keyword list. A single task can trigger multiple flags.

**Limitation:** keyword matching is imperfect at the individual task level — some words carry different meanings in different contexts (e.g., "operate" could mean operate machinery or operate a business). This noise is acceptable because these flags will be aggregated into occupation-level percentages in the next step, where individual misclassifications wash out across 10–30 tasks per occupation.

In [269]:
task_features_by_occ['has_computer']=task_features_by_occ['task_name'].str.contains(
    'compute|computer|software|database|digital|program(?:ming)?|web|network(?:ing)?|server|cloud|spreadsheet|online|internet|website|algorithm|cyber|code|coding|automat(?:e|ed|ion)|electronic.*system|information.*system|data.*system',
    case=False
    ).astype(int)

task_features_by_occ['is_physical'] = task_features_by_occ['task_name'].str.contains(r'\brepair(?:ing|s|ed)?\b|\bassembl(?:e|ing|y|ies|ed)\b|\binstall(?:ing|ation|s|ed)?\b|\bload(?:ing|s|ed)?\b|\bunload(?:ing|s|ed)?\b|\blift(?:ing|s|ed)?\b|\bweld(?:ing|s|ed)?\b|\bdrill(?:ing|s|ed)?\b|\bgrind(?:ing|s)?\b|\bshovel(?:ing|s)?\b|\bexcavat(?:e|ing|ion)\b|\bdemolish(?:ing|es|ed)?\b|\bconstruct(?:ing|ion|s|ed)?\b|\bfasten(?:ing|s|ed|er)?\b|\bbolt(?:ing|s|ed)?\b|\bsolder(?:ing|s|ed)?\b|\bhoist(?:ing|s|ed)?\b|\bhaul(?:ing|s|ed)?\b|\bpav(?:e|ing|ed)\b|\bplumb(?:ing)?\b|\bdrywall(?:ing)?\b|\bcement(?:ing)?\b|\bbrick(?:lay|laying|s)?\b|\bscaffold(?:ing)?\b|\broof(?:ing)?\b|\binsulat(?:e|ing|ion)\b|\bvacuum(?:ing|s)?\b|\bsweep(?:ing|s)?\b|\bmop(?:ping|s)?\b|\bscrub(?:bing|s)?\b|\bwash(?:ing|es|ed)?\b|\bwipe(?:s|d)?\b|\bpolish(?:ing|es|ed)?\b|\bsaniti(?:ze|zing|zed)\b|\bsteriliz(?:e|ing|ed)\b|\bclean(?:ing|s|ed)?\b|\bmow(?:ing|s|ed)?\b|\bprun(?:e|ing|ed)\b|\blandscap(?:e|ing|ed)\b|\bfertiliz(?:e|ing|ed)\b|\bfumigat(?:e|ing|ed)\b|\birrigat(?:e|ing|ed)\b|\bharvest(?:ing|s|ed)?\b|\bslaughter(?:ing|s|ed)?\b|\bbutcher(?:ing|s|ed)?\b|\bcook(?:ing|s|ed)?\b|\bbak(?:e|ing|es|ed)\b|\bgrill(?:ing|s|ed)?\b|\bfry(?:ing|ies|ied)?\b|\bcarry(?:ing|ies)?\b|\bstack(?:ing|s|ed)?\b|\bpatrol(?:ling|s|led)?\b|\brescue\b|\bextinguish(?:ing|ed)?\b|\bfirefight|\bsew(?:ing|s|n|ed)?\b|\bstitch(?:ing|es|ed)?\b|\bknit(?:ting|s|ted)?\b|\bweav(?:e|ing|es|ed)\b|\bglue\b|\bgluing\b|\bwax(?:ing)?\b|\bmassag(?:e|ing)\b|\btow(?:ing|s|ed)?\b|\b(?:arrest|apprehend|detain|handcuff|frisk)\b|\bsplint(?:ing|s|ed)?\b|\bbandage\b|\bimmobili\w+|\b(?:bathe|groom|toileting)\b|\bhand\s+tool|\bpower\s+tool|\bdisassembl(?:e|ing|y|ed)\b|\blubricat(?:e|ing|ed|ion)\b',
    case=False
).astype(int)


task_features_by_occ['must_communicate']=task_features_by_occ['task_name'].str.contains(
    'nominate|debate|propose|represent|promote|encourage|support|language|tone|warn|accept|team|child|adult|adolescent|other|director|member|crew|staff|arrange|inform|feedback|train|hire|recruit|supervise|communicat|present(?:ation)|negotiat|counsel|advise|advis(?:e|ing|ory)|teach(?:ing)?|interview|consult|collaborat|coordinat|mediat|facilitat|mentor|coach(?:ing)?|instruct|lectur|tutor|persuad|advocat|arbitrat|liais|confer|discuss|explain|translat|greet|testif|preach|officiat|client(?:s)?|customer(?:s)?|patient(?:s)?|student(?:s)?|notify|recommend|correspond|respond.*(?:inquir|request|complaint|question)',
    case=False
).astype(int)

task_features_by_occ['must_analyze']=task_features_by_occ['task_name'].str.contains(
    'goasl|deadline|knowledge|identify|market|strateg|promote|order|decision|resolve|adhere|adapt|idea|arrange|write|diagnos|analyz|research(?:ing)?|investigat|evaluat|assess|calculat|estimat|forecast|predict|statistic|audit(?:ing)?|diagnos|synthesiz|verify|validat|classif|apprais|benchmark|hypothes|feasib|(?:conduct|perform|design).*(?:stud|experiment|survey|research|test|inspection)|review|examine|inspect|compile|determin|monitor',
    case=False
).astype(int)

task_features_by_occ['must_manage']=task_features_by_occ['task_name'].str.contains(
    'direct|alert|manage|manag(?:ing|ement)|supervis|oversee|oversight|delegat|authoriz|approv(?:e|al)|budget|allocat|procurement|hire|hiring|recruit|schedul(?:e|ing)|assign|organiz|implement|establish|direct.*(?:work|activit|staff|operation)|plan.*(?:activit|project|program|operation|strateg)'
    ,case=False
).astype(int)

task_features_by_occ['is_creative']=task_features_by_occ['task_name'].str.contains(
    'marketing|warn|artwork|artistic|aesthetic|creative|illustrat|sculpt|choreograph|storyboard|lyric|photograph(?:y|ing|er|ic)|cinematograph|decorat(?:e|ing|ion|ive)|compose.*(?:music|song|melody|score|vocal)|(?:arrange|transpose|transcribe).*music|play\b.*(?:instrument|piano|guitar|violin|drum|trumpet|flute)|(?:perform|rehearse).*(?:music|dance|song|concert|recital|comedy|drama|skit)|sketch.*(?:design|drawing|art|set|idea|concept)|(?:design|create|develop).*(?:artwork|illustration|animation|graphic|logo|visual|display|layout)|write.*(?:script|story|fiction|poem|poetry|lyric|novel|press.release|advertisement|song|article|blog|editorial|commentary|column)|graphic.design|interior.design|fashion.design|costume.design|floral.design|set.design|web.design|landscape.design|(?:direct|produce|edit).*(?:film|video|movie|broadcast|production|documentary|animation)|(?:create|develop|design).*(?:recipe|menu)|photograph',
    case=False
    ).astype(int)

task_features_by_occ['is_text_native']=task_features_by_occ['task_name'].str.contains(
    'write.*(?:code|report|document|email|letter|memo|proposal|brief|summary|article|script|manual|specification|plan|policy|procedure|grant|contract|review|description|standard|guideline|instruction|recommendation)|draft.*(?:report|document|letter|memo|proposal|brief|correspondence|contract|legislation|policy|plan|budget)|prepare.*(?:report|document|correspondence|brief|presentation|statement|tax|return|invoice|budget|proposal|spreadsheet|manuscript)|compose.*(?:email|letter|memo|correspondence|message)|transcri(?:be|ption)|data.entry|enter.*data|key.*data|input.*data|edit.*(?:text|document|manuscript|copy|report|content|draft)|proofread|program(?:ming)?\b.*(?:software|computer|application|system|script|code|language)|develop.*(?:software|code|application|website|database|script|algorithm)|debug|compil(?:e|ing).*(?:data|report|record|information|statistic|document)|spreadsheet|bookkeep|ledger|translat(?:e|ing|ion).*(?:document|text|material|content)|word.process',
    case=False
    ).astype(int)


task_features_by_occ.sample(10)

,occ_code,task_name,has_computer,is_physical,must_communicate,must_analyze,must_manage,is_creative,is_text_native
336,11-3051,monitor transportation and storage of flammabl...,0,0,1,1,0,0,0
3814,17-2199,reengineer nanomaterials to improve biodegrada...,0,0,0,0,0,0,0
12779,43-3071,"carry out special services for customers, such...",0,1,1,1,0,0,0
4701,19-1042,plan and direct studies to investigate human o...,0,0,0,1,1,0,0
9892,29-2099,collect ophthalmic measurements or other diagn...,0,0,1,1,0,0,0
1213,11-9199,administer systems and programs to reduce loss...,1,0,0,0,0,0,0
17266,51-6041,estimate the costs of requested products or se...,0,1,1,1,0,0,0
7179,25-2053,"establish clear objectives for all lessons, un...",0,0,1,0,1,0,0
9370,29-1199,"adhere to local, state and federal laws, regul...",0,0,0,1,0,0,0
8547,29-1062,prepare government or organizational reports w...,0,0,0,1,1,0,1


In [270]:
task_features_by_occ[(task_features_by_occ[['has_computer', 'is_physical', 'must_communicate', 'must_analyze', 'must_manage']] == 0).all(axis=1)].task_name

27       attend and participate in meetings of municipa...
65                  plan store layouts or design displays.
72       keep abreast of the issues affecting constitue...
154      select products or accessories to be displayed...
178      develop and maintain the company's corporate i...
                               ...                        
19503    push or ride cars down slopes, or hook cars to...
19505    open and close bottom doors of cars to dump co...
19510                 maintain records of materials moved.
19519    seal outlet valves on tank cars, barges, and t...
19520    test samples for specific gravity, using hydro...
Name: task_name, Length: 4074, dtype: object

Some tasks don't match any of the five keyword categories — this is expected, since keyword matching can't capture every possible phrasing. These unclassified tasks aren't a problem: once aggregated to the occupation level, each `pct_*` feature reflects the proportion of tasks that *do* match, so unmatched tasks naturally push the percentages down rather than introducing bias.

## Aggregating task features to the occupation level

Each occupation has 10–30 individual tasks. Before merging with the anchor dataset, I collapse these into one row per `occ_code` using `groupby().agg()`.

In [271]:
occ_task_features = task_features_by_occ.groupby('occ_code').agg( 
    # keyword features
    pct_computer=('has_computer', 'mean'),
    pct_physical=('is_physical', 'mean'),
    pct_communication=('must_communicate', 'mean'),
    pct_analyze=('must_analyze', 'mean'),
    pct_manage=('must_manage', 'mean'),
    pct_creative=('is_creative', 'mean'),
    pct_textnative=('is_text_native', 'mean')
    
).reset_index()
    
occ_task_features

,occ_code,pct_computer,pct_physical,pct_communication,pct_analyze,pct_manage,pct_creative,pct_textnative
0,11-1011,0.204082,0.020408,0.530612,0.469388,0.632653,0.081633,0.102041
1,11-1021,0.117647,0.000000,0.588235,0.294118,0.705882,0.176471,0.000000
2,11-1031,0.100000,0.033333,0.466667,0.500000,0.233333,0.000000,0.066667
3,11-2011,0.121951,0.048780,0.585366,0.634146,0.341463,0.243902,0.024390
4,11-2021,0.050000,0.000000,0.600000,0.700000,0.300000,0.250000,0.000000
...,...,...,...,...,...,...,...,...
770,53-7072,0.000000,0.142857,0.142857,0.214286,0.071429,0.000000,0.000000
771,53-7073,0.000000,0.307692,0.076923,0.076923,0.153846,0.000000,0.000000
772,53-7081,0.125000,0.125000,0.250000,0.125000,0.125000,0.000000,0.000000
773,53-7111,0.000000,0.384615,0.307692,0.153846,0.153846,0.000000,0.000000


In [272]:
occ_task_features.describe()

,pct_computer,pct_physical,pct_communication,pct_analyze,pct_manage,pct_creative,pct_textnative
count,775.000000,775.000000,775.000000,775.000000,775.000000,775.000000,775.000000
mean,0.082279,0.199222,0.406641,0.324276,0.169141,0.028416,0.040516
std,0.129907,0.218274,0.227371,0.179758,0.153349,0.068176,0.060065
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.224747,0.179487,0.051282,0.000000,0.000000
50%,0.045455,0.115385,0.371429,0.333333,0.133333,0.000000,0.000000
75%,0.112698,0.333333,0.578947,0.448448,0.250000,0.030536,0.062500
max,0.933333,1.000000,1.000000,1.000000,0.800000,0.588235,0.444444


## Merging task features into the anchor dataset

To connect individual task features to occupations, I merge through `occ_code`, completing the task → occupation link.

In [273]:
df=data.copy()
df

,occ_code,title,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,ChanceAuto,observed_exposure,major_group
0,11-1011,Chief Executives,Management,0,1,5.0,189600.0,33600.0,0.25,0.0333,11
1,11-1021,General and Operations Managers,Management,1,1,4.0,100930.0,230000.0,16.00,0.1378,11
2,11-1031,Legislators,Management,0,0,4.0,24670.0,4300.0,1.35,0.0000,11
3,11-2011,Advertising and Promotions Managers,Management,0,1,1.5,117130.0,5400.0,1.50,0.1731,11
4,11-2021,Marketing Managers,Management,1,1,4.0,134290.0,26000.0,1.40,0.3195,11
...,...,...,...,...,...,...,...,...,...,...,...
751,53-7071,Gas Compressor and Gas Pumping Station Operators,Transportation and Material Moving,0,0,2.0,65210.0,400.0,91.00,0.0000,53
752,53-7072,"Pump Operators, Except Wellhead Pumpers",Transportation and Material Moving,1,0,2.0,44380.0,1600.0,90.00,0.0000,53
753,53-7073,Wellhead Pumpers,Transportation and Material Moving,0,0,2.0,53490.0,1700.0,84.00,0.0000,53
754,53-7081,Refuse and Recyclable Material Collectors,Transportation and Material Moving,1,1,2.0,37260.0,20200.0,93.00,0.0000,53


In [274]:
# merging occ codes with individual tasks
df=df.merge(occ_task_features, on='occ_code', how='left')

In [275]:
df=df[['occ_code','title','JobFamily','isBright','isGreen','JobZone','MedianSalary','JobForecast','ChanceAuto','pct_computer','pct_physical','pct_communication','pct_analyze','pct_manage','pct_creative','pct_textnative','observed_exposure','major_group']]
df['JobZone']=df['JobZone'].round().astype(int)
df.sample(5)

,occ_code,title,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,ChanceAuto,pct_computer,pct_physical,pct_communication,pct_analyze,pct_manage,pct_creative,pct_textnative,observed_exposure,major_group
649,51-4193,"Plating Machine Setters, Operators, and Tender...",Production,0,0,2,32420.0,3600.0,84.000000,0.025641,0.307692,0.076923,0.179487,0.025641,0.025641,0.000000,0.0000,51
51,13-2011,Accountants and Auditors,Business and Financial Operations,1,0,2,70500.0,438000.0,30.666667,0.166667,0.000000,0.444444,0.694444,0.361111,0.000000,0.194444,0.3478,13
251,27-1027,Set and Exhibit Designers,"Arts, Design, Entertainment, Sports, and Media",0,0,5,54270.0,1500.0,0.600000,0.000000,0.259259,0.518519,0.555556,0.185185,0.222222,0.074074,0.0000,27
391,35-9031,"Hosts and Hostesses, Restaurant, Lounge, and C...",Food Preparation and Serving Related,1,0,2,22160.0,107600.0,97.000000,0.045455,0.045455,0.500000,0.227273,0.409091,0.045455,0.000000,0.0734,35
147,19-3091,Anthropologists and Archeologists,"Life, Physical, and Social Science",1,0,3,62410.0,2400.0,6.000000,0.136364,0.090909,0.500000,0.500000,0.250000,0.090909,0.045455,0.1354,19


In [276]:
df[df.occ_code=='51-3011']

,occ_code,title,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,ChanceAuto,pct_computer,pct_physical,pct_communication,pct_analyze,pct_manage,pct_creative,pct_textnative,observed_exposure,major_group
621,51-3011,Bakers,Production,0,0,2,26520.0,28100.0,89.0,0.0,0.555556,0.277778,0.166667,0.055556,0.111111,0.0,0.0,51


In [277]:
pct_cols = ['pct_computer', 'pct_physical', 'pct_communication', 'pct_analyze', 'pct_manage', 'pct_creative', 'pct_textnative']

# imputing missing pct columns with a major group median
for col in pct_cols:
    df[col] = df.groupby('major_group')[col].transform(
        lambda x: x.fillna(x.median())
    )


print(df.isnull().sum())

occ_code             0
title                0
JobFamily            0
isBright             0
isGreen              0
JobZone              0
MedianSalary         0
JobForecast          0
ChanceAuto           0
pct_computer         0
pct_physical         0
pct_communication    0
pct_analyze          0
pct_manage           0
pct_creative         0
pct_textnative       0
observed_exposure    0
major_group          0
dtype: int64


## ChanceAuto Problem

Upon reviewing the correlation analysis and researching the source of `ChanceAuto`, I have decided to drop this variable. It was scraped from a third-party website (`willrobotstakemyjob.com`) with no documented methodology, making its provenance unverifiable. The data also contains unexplained negative values and ~14% missing entries that required imputation.

Due to the lack of supporting documentation and questionable reliability of the data source, it was decided to omit `ChanceAuto` variable for better model interpretability

In [278]:
df.drop(columns=['ChanceAuto', 'major_group'], inplace=True)
df['JobZone']=df['JobZone'].round(2)
df

,occ_code,title,JobFamily,isBright,isGreen,JobZone,MedianSalary,JobForecast,pct_computer,pct_physical,pct_communication,pct_analyze,pct_manage,pct_creative,pct_textnative,observed_exposure
0,11-1011,Chief Executives,Management,0,1,5,189600.0,33600.0,0.204082,0.020408,0.530612,0.469388,0.632653,0.081633,0.102041,0.0333
1,11-1021,General and Operations Managers,Management,1,1,4,100930.0,230000.0,0.117647,0.000000,0.588235,0.294118,0.705882,0.176471,0.000000,0.1378
2,11-1031,Legislators,Management,0,0,4,24670.0,4300.0,0.100000,0.033333,0.466667,0.500000,0.233333,0.000000,0.066667,0.0000
3,11-2011,Advertising and Promotions Managers,Management,0,1,2,117130.0,5400.0,0.121951,0.048780,0.585366,0.634146,0.341463,0.243902,0.024390,0.1731
4,11-2021,Marketing Managers,Management,1,1,4,134290.0,26000.0,0.050000,0.000000,0.600000,0.700000,0.300000,0.250000,0.000000,0.3195
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,53-7071,Gas Compressor and Gas Pumping Station Operators,Transportation and Material Moving,0,0,2,65210.0,400.0,0.000000,0.153846,0.076923,0.153846,0.000000,0.000000,0.000000,0.0000
752,53-7072,"Pump Operators, Except Wellhead Pumpers",Transportation and Material Moving,1,0,2,44380.0,1600.0,0.000000,0.142857,0.142857,0.214286,0.071429,0.000000,0.000000,0.0000
753,53-7073,Wellhead Pumpers,Transportation and Material Moving,0,0,2,53490.0,1700.0,0.000000,0.307692,0.076923,0.076923,0.153846,0.000000,0.000000,0.0000
754,53-7081,Refuse and Recyclable Material Collectors,Transportation and Material Moving,1,1,2,37260.0,20200.0,0.125000,0.125000,0.250000,0.125000,0.125000,0.000000,0.000000,0.0000


In [279]:
df.describe()

,isBright,isGreen,JobZone,MedianSalary,JobForecast,pct_computer,pct_physical,pct_communication,pct_analyze,pct_manage,pct_creative,pct_textnative,observed_exposure
count,756.000000,756.000000,756.000000,756.000000,7.560000e+02,756.000000,756.000000,756.000000,756.000000,756.000000,756.000000,756.000000,756.000000
mean,0.425926,0.165344,3.068783,56350.945767,3.046858e+04,0.067958,0.193067,0.403020,0.330301,0.168408,0.027665,0.039609,0.076977
std,0.494810,0.371737,1.248269,26992.809291,9.010565e+04,0.089866,0.216542,0.216892,0.179452,0.151060,0.067870,0.057698,0.133172
min,0.000000,0.000000,0.000000,20120.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,2.000000,36307.500000,2.200000e+03,0.000000,0.000000,0.229895,0.183612,0.052632,0.000000,0.000000,0.000000
50%,0.000000,0.000000,3.000000,50090.000000,5.300000e+03,0.044466,0.100000,0.390097,0.333333,0.125490,0.000000,0.000000,0.000000
75%,1.000000,0.000000,4.000000,71670.000000,1.845000e+04,0.100000,0.318994,0.565217,0.457440,0.250000,0.027778,0.064810,0.095675
max,1.000000,1.000000,5.000000,208000.000000,1.052000e+06,0.588235,0.900000,1.000000,1.000000,0.800000,0.588235,0.444444,0.745100


In [280]:
df.to_csv('../dataset/transformed_data.csv', index=False)